In [ ]:
# urls used :
# "https://www.coursera.org/courses?query=&refinementList%5Btopic%5D%5B0%5D=Language%20Learning&page=1&configure%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BhitsPerPage%5D=7&indices%5Btest_suggestions%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_suggestions%5D%5Bpage%5D=1&indices%5Btest_careers%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_careers%5D%5Bconfigure%5D%5BhitsPerPage%5D=1&indices%5Btest_careers%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_careers%5D%5Bpage%5D=1&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BhitsPerPage%5D=4&indices%5Btest_degrees_keyword_only%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_degrees_keyword_only%5D%5Bpage%5D=1&indices%5Btest_products%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_products%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_products%5D%5Bpage%5D=1"
# "https://www.coursera.org/courses?query=&refinementList%5Btopic%5D%5B0%5D=Computer%20Science&page=1&configure%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BhitsPerPage%5D=7&indices%5Btest_careers%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_careers%5D%5Bconfigure%5D%5BhitsPerPage%5D=1&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BhitsPerPage%5D=4&indices%5Btest_products%5D%5Bconfigure%5D%5BclickAnalytics%5D=true"
# https://www.coursera.org/learn/machine-learning/reviews

# got 48 reviews for this courses

In [ ]:

import os
from dotenv import load_dotenv
load_dotenv(".env")
api_key = os.getenv("FIRECRAWL_API_KEY")

In [24]:
from firecrawl import Firecrawl
from pydantic import BaseModel, Field
from typing import Optional

# Initialize the Firecrawl with your API key
app = Firecrawl(api_key=api_key)

# Define a proper structure for individual reviews
class Review(BaseModel):
    reviewer_name: Optional[str] = Field(description="Name of the reviewer")
    rating: float = Field(description="Rating score (e.g., 1-5 stars)")
    review_text: str = Field(description="The full text content of the review")
    date: Optional[str] = Field(description="Date when the review was posted")

# Define the main extraction schema
class ExtractSchema(BaseModel):
    reviews: list[Review] = Field(description="List of course reviews with ratings and text")

try:
    # Perform the extraction with improved prompt
    data = app.extract(
        urls = ["https://www.coursera.org/courses?query=&refinementList%5Btopic%5D%5B0%5D=Language%20Learning&page=1&configure%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BhitsPerPage%5D=7&indices%5Btest_suggestions%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_suggestions%5D%5Bpage%5D=1&indices%5Btest_careers%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_careers%5D%5Bconfigure%5D%5BhitsPerPage%5D=1&indices%5Btest_careers%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_careers%5D%5Bpage%5D=1&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BhitsPerPage%5D=4&indices%5Btest_degrees_keyword_only%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_degrees_keyword_only%5D%5Bpage%5D=1&indices%5Btest_products%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_products%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_products%5D%5Bpage%5D=1"],
        prompt='''Extract all course reviews from this page. For each review, capture:
        1. The reviewer's name (if available)
        2. The rating score (number of stars out of 5)
        3. The complete review text/comment
        4. The date the review was posted (if available)
        
        Make sure to extract the actual review content, not just metadata.''',
        schema=ExtractSchema.model_json_schema(),
        agent={'model': 'FIRE-1'}
    )
    
    # Print results in a readable format
    if data and 'data' in data:
        print(f"Total reviews extracted: {len(data['data'])}")
        print("\n" + "="*80 + "\n")
        
        for idx, review in enumerate(data['data'], 1):
            print(f"Review {idx}:")
            if 'reviews' in review:
                for r in review['reviews']:
                    print(f"  Name: {r.get('reviewer_name', 'N/A')}")
                    print(f"  Rating: {r.get('rating', 'N/A')}/5")
                    print(f"  Date: {r.get('date', 'N/A')}")
                    print(f"  Text: {r.get('review_text', 'N/A')[:200]}...")
                    print("\n" + "-"*80 + "\n")
    else:
        print("No data returned. Response:")
        print(data)
        
except Exception as e:
    print(f"Error occurred: {str(e)}")
    print("\nTrying alternative approach with scrape method...")
    
    # Alternative: Use scrape with formats if extract doesn't work
    try:
        scrape_data = app.scrape_url(
            url = "https://www.coursera.org/courses?query=&refinementList%5Btopic%5D%5B0%5D=Language%20Learning&page=1&configure%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_suggestions%5D%5Bconfigure%5D%5BhitsPerPage%5D=7&indices%5Btest_suggestions%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_suggestions%5D%5Bpage%5D=1&indices%5Btest_careers%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_careers%5D%5Bconfigure%5D%5BhitsPerPage%5D=1&indices%5Btest_careers%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_careers%5D%5Bpage%5D=1&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_degrees_keyword_only%5D%5Bconfigure%5D%5BhitsPerPage%5D=4&indices%5Btest_degrees_keyword_only%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_degrees_keyword_only%5D%5Bpage%5D=1&indices%5Btest_products%5D%5Bconfigure%5D%5BclickAnalytics%5D=true&indices%5Btest_products%5D%5BrefinementList%5D%5Bpage%5D=1&indices%5Btest_products%5D%5Bpage%5D=1",
            formats=['extract'],
            extract={
                'prompt': 'Extract all reviews with their ratings and text content',
                'schema': ExtractSchema.model_json_schema()
            }
        )
        print("Scrape result:")
        print(scrape_data)
    except Exception as scrape_error:
        print(f"Scrape error: {str(scrape_error)}")


        

No data returned. Response:
success=True id=None status='completed' data={'reviews': [{'reviewer_name': '', 'rating': 4.8, 'review_text': 'This course provided a comprehensive introduction to data analytics. I learned various skills including data visualization and storytelling, which I could apply directly in my job.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'The Google Cybersecurity course was excellent! It covered essential topics and gave lots of practical insights. Highly recommend for anyone entering the field.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'Fantastic course on project management! The instructors were knowledgeable and the content was very relevant to real-world applications.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'This specialization on AI essentials is a must for those who want to understand artificial intelligence concepts without getting too technical.', 'date': ''}, {'reviewer_name': ''

In [ ]:
# take the data from the response output above
data={'reviews': [{'reviewer_name': '', 'rating': 4.8, 'review_text': 'This course provided a comprehensive introduction to data analytics. I learned various skills including data visualization and storytelling, which I could apply directly in my job.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'The Google Cybersecurity course was excellent! It covered essential topics and gave lots of practical insights. Highly recommend for anyone entering the field.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'Fantastic course on project management! The instructors were knowledgeable and the content was very relevant to real-world applications.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'This specialization on AI essentials is a must for those who want to understand artificial intelligence concepts without getting too technical.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': "The Digital Marketing & E-commerce course offered great insights into marketing strategies and tactics. I'm already seeing improvements in my campaigns!", 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'Highly beneficial! The IT support course was well-paced and gave practical skills that are directly applicable to real-world scenarios.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'The UX Design course broke down the design process into easy-to-follow steps. I learned a lot about user-centered design principles.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': 'The Prompting Essentials course was very informative. I appreciated the focus on generative AI training techniques.', 'date': ''}, {'reviewer_name': '', 'rating': 4.9, 'review_text': 'Machine Learning made easy! This course demystified complex topics and provided a solid foundation in ML frameworks.', 'date': ''}, {'reviewer_name': '', 'rating': 4.6, 'review_text': 'The IBM Generative AI Engineering course was insightful. It covered a broad range of topics and had great hands-on components.', 'date': ''}, {'reviewer_name': '', 'rating': 4.8, 'review_text': "AI for Everyone provided a solid introduction to AI's impact on different industries. Very engaging and informative.", 'date': ''}, {'reviewer_name': '', 'rating': 4.7, 'review_text': 'The Advanced Data Analytics course helped me refine my skills in data storytelling and visualization. Great for advancing my career!', 'date': ''}]}



import json
import os

# Path to reviews.json - adjust path if needed
reviews_json_path = "./real_reviews.json"

# Load existing reviews from file
if os.path.exists(reviews_json_path):
    with open(reviews_json_path, "r", encoding="utf-8") as f:
        try:
            existing_data = json.load(f)
        except Exception:
            existing_data = {}
else:
    existing_data = {}

# Ensure the base structure is correct
if "reviews" not in existing_data or not isinstance(existing_data["reviews"], list):
    existing_data["reviews"] = []

# Add new reviews (from variable 'data')
for review in data.get("reviews", []):
    existing_data["reviews"].append(review)

# Write back to file
with open(reviews_json_path, "w", encoding="utf-8") as f:
    json.dump(existing_data, f, indent=2, ensure_ascii=False)

print(f"Added {len(data.get('reviews', []))} new reviews to {reviews_json_path}")


Added 12 new reviews to reviews.json


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_json("./real_reviews.jsonl")
print(len(df))

48
